# Project 16 — Dynamic Month-End Finance Agents

This compact notebook reproduces the **agentic core** of the backend:

1. load finance evidence and policies;
2. expose deterministic finance calculations as tools;
3. create live OpenAI-powered specialist agents;
4. let an orchestrator choose agents and repeat them at runtime;
5. pause for human approval when the model identifies a material action;
6. evaluate the final result against evidence.

It deliberately leaves FastAPI, Streamlit, persistence, and deployment in
the application project.  Those layers do not help explain agent reasoning.

**Only live model execution is supported.** Running the agent cells requires
a valid `OPENAI_API_KEY` in the project `.env` file.

## 1. Imports and environment

`load_dotenv` reads secrets without placing them in the notebook.
LangGraph supplies the state machine, message reducer, checkpoint memory,
sub-agent factory, and interrupt/resume primitives.

In [ ]:
# CELL PURPOSE: Import only the libraries used by the agent demonstration.
import json
import os
import re
import uuid
from pathlib import Path
from typing import Annotated, Any, Literal, TypedDict

import httpx
import pandas as pd
from dotenv import load_dotenv
from IPython.display import Image, display
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import create_react_agent
from langgraph.types import Command, interrupt
from pydantic import BaseModel, Field

# The notebook lives in <project>/notebooks, so its parent is the project root.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(PROJECT_ROOT / ".env")

# Fail clearly instead of silently switching to a fake or deterministic model.
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("Add OPENAI_API_KEY to the project .env before running this notebook.")

MODEL_NAME = os.getenv("OPENAI_MODEL", "gpt-5-mini")
# GPT-5-family models currently use their supported default temperature.
# Passing temperature=0 would make the API reject the request.
model = ChatOpenAI(model=MODEL_NAME)
print(f"Live model configured: {MODEL_NAME}")

## 2. Load the evidence pack

The loader discovers local CSV, JSON, TXT, and Markdown files. The agents
receive a compact catalogue, while tools read the full data. This mirrors a
production design where prompts carry metadata and tools fetch exact evidence.

In [ ]:
# CELL PURPOSE: Discover the synthetic month-end evidence without app imports.
DATA_ROOT = PROJECT_ROOT / "data"
if not DATA_ROOT.exists():
    raise FileNotFoundError(f"Expected the project data folder at {DATA_ROOT}")

tables: dict[str, pd.DataFrame] = {}
documents: dict[str, str] = {}
ground_truth: dict[str, Any] = {}

for path in sorted(DATA_ROOT.rglob("*")):
    if not path.is_file():
        continue
    relative_name = path.relative_to(DATA_ROOT).as_posix()
    # Hidden labels are isolated from operational agents. Only the
    # post-run Evaluation Agent is allowed to receive this mapping.
    if "ground_truth" in Path(relative_name).parts:
        if path.suffix.lower() == ".json":
            ground_truth[relative_name] = json.loads(path.read_text(encoding="utf-8"))
        continue
    if path.suffix.lower() == ".csv":
        tables[relative_name] = pd.read_csv(path)
    elif path.suffix.lower() == ".json":
        documents[relative_name] = json.dumps(json.loads(path.read_text(encoding="utf-8")), indent=2)
    elif path.suffix.lower() in {".txt", ".md"}:
        documents[relative_name] = path.read_text(encoding="utf-8")

data_catalog = {
    name: {"rows": len(frame), "columns": frame.columns.tolist()}
    for name, frame in tables.items()
}
print(f"Loaded {len(tables)} tables, {len(documents)} operational documents, "
      f"and {len(ground_truth)} isolated evaluation files.")
display(pd.DataFrame.from_dict(data_catalog, orient="index"))

## 3. Agent Cards and the A2A boundary

An Agent Card is the orchestrator's capability catalogue. It describes an
agent's identity, skills, operating mode, and endpoint. The orchestrator may
select only registered business agents. Guard and Evaluation are mandatory
control agents, so the orchestrator cannot skip them.

Valuation is intentionally different: it is independently deployed and reached
through A2A. The notebook first discovers its public card and then sends a
JSON-RPC `message/send` task. The remote graph, prompts, and tools remain private.

In [ ]:
# CELL PURPOSE: Publish the supervisor-visible capability registry.
A2A_BASE_URL = os.getenv("VALUATION_AGENT_URL", "http://localhost:9000").rstrip("/")
A2A_VERSION = os.getenv("A2A_VERSION", "1.0")

AGENT_CARDS = {
    "reconciliation": {
        "name": "Reconciliation Agent", "mode": "local_langgraph",
        "description": "Finds position, cash, transaction, custody, and settlement breaks.",
        "skills": ["position reconciliation", "cash reconciliation", "duplicate detection"],
    },
    "accounting": {
        "name": "Accounting Agent", "mode": "local_langgraph",
        "description": "Investigates ledger, income, fee, accrual, and journal issues.",
        "skills": ["ledger review", "fee accrual", "income review", "journal recommendation"],
    },
    "risk_policy": {
        "name": "Risk and Policy Agent", "mode": "local_langgraph",
        "description": "Applies mandate, materiality, restricted-security, and approval policies.",
        "skills": ["mandate checks", "materiality", "restricted holdings", "approval routing"],
    },
    "valuation": {
        "name": "Remote Valuation Agent", "mode": "remote_a2a",
        "description": "Investigates stale prices, source conflicts, fair value, and NAV impact.",
        "skills": ["stale-price review", "price challenge", "NAV impact", "fair value"],
        "agent_card_url": f"{A2A_BASE_URL}/.well-known/agent-card.json",
    },
    "evidence_critic": {
        "name": "Evidence Critic Agent", "mode": "local_langgraph",
        "description": "Challenges citations, completeness, ownership, and approval readiness.",
        "skills": ["evidence validation", "citation review", "audit readiness"],
    },
    "guard": {
        "name": "Guard Agent", "mode": "mandatory_control",
        "description": "Validates every proposed delegation before execution.",
        "skills": ["allow-list enforcement", "scope review", "approval escalation"],
    },
    "evaluation": {
        "name": "Evaluation Agent", "mode": "post_run_isolated",
        "description": "Scores the completed run against hidden ground truth and trace evidence.",
        "skills": ["exception recall", "evidence quality", "routing quality", "regression review"],
    },
}

display(pd.DataFrame.from_dict(AGENT_CARDS, orient="index")[["name", "mode", "description"]])

In [ ]:
# CELL PURPOSE: Discover and display the actual remote A2A Agent Card.
def discover_valuation_agent_card() -> dict[str, Any]:
    '''Fetch the remote identity, protocol version, endpoint, modes, and skills.'''
    with httpx.Client(timeout=httpx.Timeout(4.0, connect=1.0)) as client:
        response = client.get(f"{A2A_BASE_URL}/.well-known/agent-card.json")
        response.raise_for_status()
        card = response.json()
    required = {"name", "version", "protocolVersion", "url", "skills"}
    missing = sorted(required - set(card))
    if missing:
        raise ValueError(f"Remote Agent Card is missing required fields: {missing}")
    return card


try:
    remote_valuation_card = discover_valuation_agent_card()
    display(remote_valuation_card)
except Exception as exc:
    remote_valuation_card = None
    print("Remote valuation service is not running yet.")
    print("Start it with: uvicorn remote_agents.valuation_agent.app:app --port 9000")
    print("Discovery error:", exc)

In [ ]:
# CELL PURPOSE: Implement the notebook's real A2A JSON-RPC client.
def send_valuation_a2a(task: str, existing_findings: list[dict[str, Any]]) -> tuple[dict, dict]:
    '''Discover the remote card, send message/send, and normalize its artifact and trace.'''
    card = discover_valuation_agent_card()
    request_id = f"req-{uuid.uuid4().hex}"
    message_id = f"msg-{uuid.uuid4().hex}"
    task_data = {
        "run_id": "notebook-class-demo", "portfolio_id": None,
        "close_period": "2026-10-31", "task": task,
        "existing_exceptions": existing_findings,
    }
    payload = {
        "jsonrpc": "2.0", "id": request_id, "method": "message/send",
        "params": {"message": {"role": "user", "messageId": message_id,
                                 "parts": [{"kind": "text", "text": json.dumps(task_data)}]}},
    }
    headers = {"A2A-Version": A2A_VERSION, "Content-Type": "application/json"}
    with httpx.Client(timeout=httpx.Timeout(120.0, connect=2.0)) as client:
        response = client.post(card["url"], json=payload, headers=headers)
        response.raise_for_status()
        body = response.json()
    if "error" in body:
        raise RuntimeError(f"A2A JSON-RPC error: {body['error']}")
    task_object = body["result"]
    artifact = task_object["artifacts"][0]
    output = json.loads(artifact["parts"][0]["text"])
    trace = {"agent": card["name"], "version": card["version"],
             "request_id": request_id, "task_id": task_object["id"],
             "context_id": task_object["contextId"],
             "state": task_object["status"]["state"], "endpoint": card["url"]}
    return output, trace

## 4. Finance tools

The LLM decides **which tool to call and with which arguments**. The tools
themselves remain deterministic because arithmetic, filtering, and evidence
lookup must be reproducible and auditable.

- `inspect_dataset`: inspect schema and representative rows before reasoning.
- `compare_datasets`: reconcile the same business key across two sources.
- `numeric_summary`: aggregate a metric by portfolio, account, or security.
- `flag_threshold_breaches`: locate material exceptions using an explicit rule.
- `search_policy`: retrieve policy evidence that governs an exception.

In [ ]:
# CELL PURPOSE: Add small helpers used by several finance tools.
def resolve_table(file_hint: str) -> tuple[str, pd.DataFrame]:
    '''Resolve a model-supplied partial filename to exactly one loaded table.'''
    hint = file_hint.strip().lower()
    matches = [(name, frame) for name, frame in tables.items() if hint in name.lower()]
    if not matches:
        raise ValueError(f"No table matches {file_hint!r}. Available: {list(tables)}")
    if len(matches) > 1:
        raise ValueError(f"Ambiguous table hint {file_hint!r}. Matches: {[name for name, _ in matches]}")
    return matches[0]


def require_columns(frame: pd.DataFrame, *columns: str) -> None:
    '''Stop a tool call early when the model invented or misspelled a column.'''
    missing = [column for column in columns if column not in frame.columns]
    if missing:
        raise ValueError(f"Missing columns {missing}. Available: {frame.columns.tolist()}")

In [ ]:
# CELL PURPOSE: Define evidence inspection and reconciliation tools.
@tool
def inspect_dataset(file_hint: str, row_limit: int = 5) -> str:
    '''Return schema and sample rows for one finance dataset.

    file_hint: Full or partial CSV filename from the catalogue.
    row_limit: Number of representative rows; capped to prevent huge prompts.
    '''
    name, frame = resolve_table(file_hint)
    limit = max(1, min(row_limit, 10))
    payload = {"file": name, "rows": len(frame), "columns": frame.columns.tolist(),
               "sample": frame.head(limit).fillna("").to_dict(orient="records")}
    return json.dumps(payload, default=str)


@tool
def compare_datasets(
    left_file: str, right_file: str, key_column: str,
    left_value_column: str, right_value_column: str,
    tolerance: float = 0.01,
) -> str:
    '''Reconcile numeric values by key and return only breaks above tolerance.'''
    left_name, left = resolve_table(left_file)
    right_name, right = resolve_table(right_file)
    key_columns = [item.strip() for item in key_column.split(",") if item.strip()]
    if not key_columns:
        raise ValueError("key_column must contain at least one column name")
    require_columns(left, *key_columns, left_value_column)
    require_columns(right, *key_columns, right_value_column)
    left_values = left[[*key_columns, left_value_column]].rename(columns={left_value_column: "left_value"})
    right_values = right[[*key_columns, right_value_column]].rename(columns={right_value_column: "right_value"})
    merged = left_values.merge(right_values, on=key_columns, how="outer")
    merged["difference"] = (
        pd.to_numeric(merged["left_value"], errors="coerce").fillna(0)
        - pd.to_numeric(merged["right_value"], errors="coerce").fillna(0)
    )
    breaks = merged[merged["difference"].abs() > abs(tolerance)]
    return json.dumps({"left": left_name, "right": right_name, "break_count": len(breaks),
                       "breaks": breaks.head(50).fillna("").to_dict(orient="records")}, default=str)

In [ ]:
# CELL PURPOSE: Define aggregation, exception detection, and policy retrieval tools.
@tool
def numeric_summary(file_hint: str, group_by: str, value_column: str) -> str:
    '''Sum a numeric column by a business dimension and return ranked totals.'''
    name, frame = resolve_table(file_hint)
    require_columns(frame, group_by, value_column)
    working = frame.copy()
    working[value_column] = pd.to_numeric(working[value_column], errors="coerce").fillna(0)
    result = working.groupby(group_by, dropna=False)[value_column].sum().sort_values(ascending=False)
    return json.dumps({"file": name, "summary": result.head(50).to_dict()}, default=str)


@tool
def flag_threshold_breaches(
    file_hint: str, metric_column: str, threshold: float,
    direction: Literal["above", "below", "absolute_above"] = "absolute_above",
) -> str:
    '''Return rows breaching an explicit numeric threshold.'''
    name, frame = resolve_table(file_hint)
    require_columns(frame, metric_column)
    metric = pd.to_numeric(frame[metric_column], errors="coerce")
    mask = metric > threshold if direction == "above" else metric < threshold
    if direction == "absolute_above":
        mask = metric.abs() > abs(threshold)
    breaches = frame[mask.fillna(False)]
    return json.dumps({"file": name, "rule": f"{direction} {threshold}", "count": len(breaches),
                       "rows": breaches.head(50).fillna("").to_dict(orient="records")}, default=str)


@tool
def search_policy(query: str, top_k: int = 3) -> str:
    '''Retrieve the most relevant policy passages using transparent lexical scoring.'''
    terms = set(re.findall(r"[a-z0-9]+", query.lower()))
    ranked = []
    for name, text in documents.items():
        chunks = [part.strip() for part in re.split(r"\n\s*\n", text) if part.strip()]
        for index, chunk in enumerate(chunks):
            words = set(re.findall(r"[a-z0-9]+", chunk.lower()))
            score = len(terms & words) / max(1, len(terms))
            ranked.append((score, name, index, chunk[:1800]))
    ranked.sort(reverse=True)
    hits = [{"score": round(score, 3), "source": name, "section": index, "text": chunk}
            for score, name, index, chunk in ranked[:max(1, min(top_k, 6))]]
    return json.dumps(hits)

## 5. Detailed LangGraph agents

Each business specialist is a LangGraph ReAct agent. It can inspect evidence,
choose tools, observe results, revise its reasoning, and call another tool.
Valuation is absent from this local list because it runs behind the A2A boundary.

The Evidence Critic is an operational agent: it challenges the package before
approval. The Guard and Evaluation agents are defined separately because they
are mandatory controls rather than optional business delegates.

In [ ]:
# CELL PURPOSE: Describe specialist capabilities and the tools each may use.
SPECIALIST_SPECS = {
    "reconciliation": {
        "purpose": "Find unexplained breaks between accounting, custodian, cash, and position records.",
        "tools": [inspect_dataset, compare_datasets, numeric_summary, search_policy],
    },
    "accounting": {
        "purpose": "Investigate ledger, income, fee, accrual, and journal exceptions.",
        "tools": [inspect_dataset, compare_datasets, numeric_summary, search_policy],
    },
    "risk_policy": {
        "purpose": "Test mandates, concentration, restricted holdings, materiality, and approval policy.",
        "tools": [inspect_dataset, numeric_summary, flag_threshold_breaches, search_policy],
    },
    "evidence_critic": {
        "purpose": "Challenge unsupported claims, missing ownership, and approval readiness.",
        "tools": [inspect_dataset, search_policy],
    },
}

catalogue_text = json.dumps(data_catalog, indent=2)
specialist_agents = {}
for name, spec in SPECIALIST_SPECS.items():
    prompt = f'''You are the {name} specialist for an investment-management month-end close.
    Mission: {spec['purpose']}
    Available data catalogue:\n{catalogue_text}
    Use tools to verify claims. Report materiality, evidence filenames, unresolved uncertainty,
    and a recommended next action. Never invent a column, value, policy, or approval.'''
    specialist_agents[name] = create_react_agent(model, spec["tools"], prompt=prompt)

agent_detail_rows = []
for name, graph_agent in specialist_agents.items():
    agent_detail_rows.append({
        "agent": name, "mode": AGENT_CARDS[name]["mode"],
        "mission": SPECIALIST_SPECS[name]["purpose"],
        "tools": [item.name for item in SPECIALIST_SPECS[name]["tools"]],
    })
display(pd.DataFrame(agent_detail_rows))

### Mandatory Guard Agent

The Guard Agent receives every proposed delegation. It does not perform the
finance task. It checks whether the target is registered, the requested action
is in scope, sensitive operations need approval, and available evidence is
sufficient. Its structured result is `allow`, `review`, or `block`.

In [ ]:
# CELL PURPOSE: Build a model-driven Guard Agent as its own LangGraph subgraph.
class GuardDecision(BaseModel):
    decision: Literal["allow", "review", "block"]
    reason: str
    checks: list[str]
    required_approval: bool


class GuardState(TypedDict, total=False):
    proposed_action: dict[str, Any]
    prior_findings: list[dict[str, Any]]
    result: dict[str, Any]


guard_model = model.with_structured_output(GuardDecision)

def guard_review_node(state: GuardState) -> dict[str, Any]:
    proposed = state["proposed_action"]
    registered = proposed.get("specialist") in AGENT_CARDS
    result = guard_model.invoke([HumanMessage(content=f'''Act as the mandatory finance-agent guard.
    Registered target: {registered}
    Proposed action: {json.dumps(proposed)}
    Existing findings: {json.dumps(state.get('prior_findings', []), default=str)}
    Allow ordinary read-only investigation. Require review for journals, overrides,
    publication, acceptance of material exceptions, or uncertain scope. Block unknown
    agents, attempts to bypass approval, or requests unrelated to month-end review.''')])
    return {"result": result.model_dump()}


guard_builder = StateGraph(GuardState)
guard_builder.add_node("policy_guard", guard_review_node)
guard_builder.add_edge(START, "policy_guard")
guard_builder.add_edge("policy_guard", END)
guard_agent = guard_builder.compile()
print("Guard Agent compiled as a mandatory LangGraph subgraph.")

### Isolated Evaluation Agent

Evaluation runs **after** the operational workflow. Hidden labels are never
exposed to the orchestrator or specialists. The agent reviews deterministic
metrics plus the completed trace, then explains failure modes and regression risk.

In [ ]:
# CELL PURPOSE: Build the post-run Evaluation Agent as a separate LangGraph graph.
class EvaluationResult(BaseModel):
    verdict: Literal["pass", "needs_review", "fail"]
    strengths: list[str]
    gaps: list[str]
    recommended_tests: list[str]


class EvaluationState(TypedDict, total=False):
    metrics: dict[str, Any]
    trace: list[dict[str, Any]]
    final_answer: str
    result: dict[str, Any]


evaluation_model = model.with_structured_output(EvaluationResult)

def evaluation_review_node(state: EvaluationState) -> dict[str, Any]:
    result = evaluation_model.invoke([HumanMessage(content=f'''You are an independent AI evaluation agent.
    Metrics derived from hidden labels: {json.dumps(state['metrics'], default=str)}
    Execution trace: {json.dumps(state.get('trace', []), default=str)}
    Final answer: {state.get('final_answer', '')}
    Judge evidence grounding, exception coverage, routing, approval handling, and bounded execution.
    Explain gaps without changing the operational result.''')])
    return {"result": result.model_dump()}


evaluation_builder = StateGraph(EvaluationState)
evaluation_builder.add_node("independent_evaluator", evaluation_review_node)
evaluation_builder.add_edge(START, "independent_evaluator")
evaluation_builder.add_edge("independent_evaluator", END)
evaluation_agent = evaluation_builder.compile()
print("Evaluation Agent compiled and isolated from operational routing.")

## 6. Shared state and dynamic orchestrator decision

The state is the shared case file. The orchestrator sees earlier findings and
decides at runtime whether to call a specialist, call one again with a refined
task, request approval, or finish. `max_rounds` is only a safety budget—it does
not define the route.

In [ ]:
# CELL PURPOSE: Define the shared LangGraph state and structured router output.
AgentName = Literal["reconciliation", "accounting", "risk_policy", "valuation", "evidence_critic"]


class OrchestratorDecision(BaseModel):
    '''Schema that prevents free-form model text from becoming graph control flow.'''
    action: Literal["delegate", "request_approval", "finish"]
    specialist: AgentName | None = None
    task: str = Field(description="Specific instruction for the selected specialist or reviewer.")
    rationale: str = Field(description="Why this action is useful given current evidence.")
    confidence: float = Field(ge=0, le=1)


class FinanceState(TypedDict, total=False):
    messages: Annotated[list[Any], add_messages]
    request: str
    findings: list[dict[str, Any]]
    trace: list[dict[str, Any]]
    rounds: int
    max_rounds: int
    decision: dict[str, Any]
    guard_result: dict[str, Any]
    a2a_tasks: list[dict[str, Any]]
    approval_granted: bool
    final_answer: str

In [ ]:
# CELL PURPOSE: Let the live model choose the next agent from state, not from hard-coded rules.
router_model = model.with_structured_output(OrchestratorDecision)


def decide_next(state: FinanceState) -> dict[str, Any]:
    '''Ask the orchestrator model what should happen next based on accumulated evidence.'''
    rounds = state.get("rounds", 0)
    max_rounds = state.get("max_rounds", 8)
    if rounds >= max_rounds:
        # A deterministic limit prevents runaway cost; it does not choose the business route.
        decision = OrchestratorDecision(
            action="finish", task="Summarize current evidence and unresolved items.",
            rationale="The execution safety budget has been reached.", confidence=0.5,
        )
    else:
        decision = router_model.invoke([
            HumanMessage(content=f'''You orchestrate a month-end portfolio review.
            User request: {state['request']}
            Registered business Agent Cards: {json.dumps({k: v for k, v in AGENT_CARDS.items() if k not in {'guard', 'evaluation'}}, default=str)}
            Prior specialist findings: {json.dumps(state.get('findings', []), default=str)}
            Approval already granted: {state.get('approval_granted', False)}
            Choose the next action. Delegate only when another specialist can add evidence.
            You may call the same specialist again with a refined task. Request human approval
            before accepting, publishing, or overriding a material exception. Finish when the
            evidence is sufficient for a grounded close recommendation.''')
        ])

    event = {"step": rounds + 1, "node": "orchestrator", **decision.model_dump()}
    return {"decision": decision.model_dump(), "trace": [*state.get("trace", []), event]}

In [ ]:
# CELL PURPOSE: Run the mandatory Guard Agent before any specialist delegation.
def run_guard_agent(state: FinanceState) -> dict[str, Any]:
    '''Ask the independent guard subgraph whether the proposed delegation may execute.'''
    proposed = {**state["decision"], "approval_granted": state.get("approval_granted", False)}
    guard_output = guard_agent.invoke({
        "proposed_action": proposed,
        "prior_findings": state.get("findings", []),
    })["result"]
    updated_decision = dict(state["decision"])
    if guard_output["decision"] in {"review", "block"}:
        updated_decision.update({
            "action": "request_approval",
            "task": f"Guard escalation: {guard_output['reason']}",
            "rationale": guard_output["reason"],
        })
    event = {"node": "guard", **guard_output,
             "target": state["decision"].get("specialist")}
    return {"guard_result": guard_output, "decision": updated_decision,
            "trace": [*state.get("trace", []), event]}

In [ ]:
# CELL PURPOSE: Execute whichever specialist the orchestrator selected at runtime.
def run_selected_specialist(state: FinanceState) -> dict[str, Any]:
    '''Invoke a local ReAct graph or the independently hosted A2A valuation graph.'''
    decision = state["decision"]
    specialist = decision.get("specialist")
    if specialist not in AGENT_CARDS or specialist in {"guard", "evaluation"}:
        raise ValueError(f"The delegate action requires a valid specialist; received {specialist!r}")

    a2a_tasks = list(state.get("a2a_tasks", []))
    if specialist == "valuation":
        output, remote_trace = send_valuation_a2a(decision["task"], state.get("findings", []))
        report = json.dumps(output, default=str)
        a2a_tasks.append(remote_trace)
        transport = "A2A JSON-RPC"
    else:
        result = specialist_agents[specialist].invoke({
            "messages": [HumanMessage(content=decision["task"])]
        })
        report = result["messages"][-1].content
        transport = "local LangGraph"

    finding = {"specialist": specialist, "transport": transport,
               "task": decision["task"], "report": report}
    event = {"step": state.get("rounds", 0) + 1, "node": specialist,
             "transport": transport, "summary": str(report)[:500]}
    return {
        "findings": [*state.get("findings", []), finding],
        "trace": [*state.get("trace", []), event],
        "a2a_tasks": a2a_tasks,
        "rounds": state.get("rounds", 0) + 1,
        "messages": [AIMessage(content=f"{specialist} report:\n{report}")],
    }

## 7. Human approval and final response

`interrupt` checkpoints the full shared state. A reviewer can inspect the
evidence and later resume the same thread with an explicit decision. The LLM
may recommend an action, but it cannot approve its own material exception.

In [ ]:
# CELL PURPOSE: Pause a material action and convert a human response into state.
def request_human_approval(state: FinanceState) -> dict[str, Any]:
    '''Suspend execution until a human supplies approve=True or approve=False.'''
    response = interrupt({
        "question": state["decision"]["task"],
        "rationale": state["decision"]["rationale"],
        "findings": state.get("findings", []),
    })
    approved = bool(response.get("approve", False)) if isinstance(response, dict) else False
    event = {"node": "human_approval", "approved": approved, "response": response}
    return {"approval_granted": approved, "trace": [*state.get("trace", []), event],
            "rounds": state.get("rounds", 0) + 1}


def write_final_answer(state: FinanceState) -> dict[str, Any]:
    '''Produce the executive close recommendation from evidence already collected.'''
    response = model.invoke([
        HumanMessage(content=f'''Prepare a concise month-end portfolio review.
        Original request: {state['request']}
        Findings: {json.dumps(state.get('findings', []), default=str)}
        Approval granted: {state.get('approval_granted', False)}
        Include status, material exceptions, evidence sources, approval state,
        unresolved risks, and next actions. Do not add facts not present above.''')
    ])
    return {"final_answer": str(response.content), "messages": [response]}

In [ ]:
# CELL PURPOSE: Convert the model's structured action into the next graph edge.
def route_decision(state: FinanceState) -> str:
    '''Map a validated action to a node name; reject malformed model output.'''
    action = state["decision"]["action"]
    routes = {"delegate": "guard", "request_approval": "approval", "finish": "finalize"}
    if action not in routes:
        raise ValueError(f"Unsupported orchestrator action: {action}")
    return routes[action]


def route_guard(state: FinanceState) -> str:
    '''Allow safe work or route review/block decisions to accountable approval.'''
    return "specialist" if state["guard_result"]["decision"] == "allow" else "approval"


builder = StateGraph(FinanceState)
builder.add_node("orchestrator", decide_next)
builder.add_node("guard", run_guard_agent)
builder.add_node("specialist", run_selected_specialist)
builder.add_node("approval", request_human_approval)
builder.add_node("finalize", write_final_answer)
builder.add_edge(START, "orchestrator")
builder.add_conditional_edges("orchestrator", route_decision,
                              {"guard": "guard", "approval": "approval", "finalize": "finalize"})
builder.add_conditional_edges("guard", route_guard,
                              {"specialist": "specialist", "approval": "approval"})
builder.add_edge("specialist", "orchestrator")
builder.add_edge("approval", "orchestrator")
builder.add_edge("finalize", END)

# MemorySaver is in-process checkpoint memory for the classroom demo.
graph = builder.compile(checkpointer=MemorySaver())
print("Dynamic graph compiled.")

In [ ]:
# CELL PURPOSE: Display the graph learners are about to execute.
try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    # Mermaid text remains useful when the optional rendering service is unavailable.
    print(graph.get_graph().draw_mermaid())

## 8. Start a real dynamic review

Change the request to observe a different route. There is no preselected first
specialist. The orchestrator may repeat agents, ask another specialist to
challenge a finding, pause for approval, or finish early.

In [ ]:
# CELL PURPOSE: Start one checkpointed case and stream visible node updates.
thread_config = {"configurable": {"thread_id": "class-demo-001"}}
initial_state: FinanceState = {
    "messages": [HumanMessage(content="Review the month-end portfolio close and identify material exceptions.")],
    "request": "Review reconciliation, accounting, mandate, and valuation evidence; investigate material breaks and prepare an approval-ready recommendation.",
    "findings": [],
    "trace": [],
    "a2a_tasks": [],
    "rounds": 0,
    "max_rounds": 8,
    "approval_granted": False,
}

latest_update = None
for update in graph.stream(initial_state, config=thread_config, stream_mode="updates"):
    latest_update = update
    print("NODE UPDATE:", list(update))
    display(update)

In [ ]:
# CELL PURPOSE: Inspect why the agent stopped before deciding whether to resume it.
snapshot = graph.get_state(thread_config)
print("Next node(s):", snapshot.next)
print("Executed rounds:", snapshot.values.get("rounds"))
display(pd.DataFrame(snapshot.values.get("trace", [])))
print("A2A remote task correlations:")
display(pd.DataFrame(snapshot.values.get("a2a_tasks", [])))

In [ ]:
# CELL PURPOSE: Resume only when the graph is waiting at a human-approval interrupt.
# Change True to False to demonstrate rejection and the orchestrator's follow-up behavior.
if snapshot.next:
    for update in graph.stream(
        Command(resume={"approve": True, "reviewer": "portfolio_controller", "comment": "Evidence reviewed."}),
        config=thread_config,
        stream_mode="updates",
    ):
        print("RESUMED NODE:", list(update))
        display(update)
else:
    print("The agent finished without requesting approval.")

## 9. Inspect and evaluate the completed case

Agent evaluation is not just answer similarity. First, deterministic metrics
compare discovered exception IDs and called agents with hidden labels. Then the
isolated Evaluation Agent interprets those metrics, trace, and final answer.

In [ ]:
# CELL PURPOSE: Compute hidden-label metrics, then ask the isolated Evaluation Agent to review them.
completed = graph.get_state(thread_config).values
final_answer = completed.get("final_answer", "")
findings = completed.get("findings", [])
trace = completed.get("trace", [])

labels = next(iter(ground_truth.values()), [])
expected_ids = {item["exception_id"] for item in labels}
expected_agents = {agent for item in labels for agent in item.get("expected_agents", [])}
evidence_text = json.dumps(findings, default=str)
found_ids = {exception_id for exception_id in expected_ids if exception_id in evidence_text}
called_agents = {item.get("node") for item in trace if item.get("node") in AGENT_CARDS}
approval_expected = any(item.get("approval_required") for item in labels)

metrics = {
    "has_final_answer": bool(final_answer.strip()),
    "exception_recall": len(found_ids) / max(1, len(expected_ids)),
    "agent_coverage": len(called_agents & expected_agents) / max(1, len(expected_agents)),
    "approval_compliance": (not approval_expected) or bool(completed.get("approval_granted")),
    "bounded_execution": completed.get("rounds", 0) <= completed.get("max_rounds", 8) + 1,
    "a2a_valuation_used": any(item.get("transport") == "A2A JSON-RPC" for item in findings),
    "guard_executed": any(item.get("node") == "guard" for item in trace),
}

print(final_answer or "The case is still waiting for another resume step.")
display(pd.Series(metrics, name="deterministic_result"))

evaluation = evaluation_agent.invoke({
    "metrics": metrics, "trace": trace, "final_answer": final_answer,
})["result"]
display(evaluation)

## What is dynamic, and what intentionally is not?

**Dynamic agent behavior**

- the orchestrator chooses the next specialist from current evidence;
- it may call the same specialist more than once with a refined task;
- each specialist chooses its own tools and arguments;
- the Guard Agent dynamically assesses the proposed delegation;
- valuation executes remotely through its discovered A2A Agent Card;
- the isolated Evaluation Agent judges metrics and trace quality;
- the orchestrator decides when evidence is sufficient or approval is needed.

**Deterministic production controls**

- reconciliation arithmetic and thresholds;
- tool input validation and bounded output;
- the maximum agent-round budget;
- human authority over material approvals.

This combination is deliberate: use agents for ambiguous judgment and routing,
but keep money, permissions, and safety controls reproducible.